# Solving a PDE Using a DeepONet

In this notebook we are going to implement a DeepONet, which it was presented in Lu, Jin, Pang, Zhang & Karniadakis (2021), *Nature Machine Intelligence 3*, 218–229, to learn the solution operator of a PDE.

The difference with a PINN is that a PINN solves one specific problem (one initial condition) and we need to train it again if the problem changes. Here we want to learn the operator
$$
    G: u_0 \longmapsto u(x, t),
$$
this is, a model that receives the initial condition $u_0$ as an input function and returns the solution of the PDE in any point $(x, t)$. Once the network is trained, solving a new initial condition is only one forward pass.

For this we represent $u_0$ by its values in $m$ fixed sensor points and we give them to the DeepONet together with the query point $(x, t)$. To train the model we generate many initial conditions with their exact solutions, and we minimize the MSE between the prediction of the network and the exact solution:
$$
    \mathcal{L}_{MSE} = \frac{1}{N_{u_0}\,N_{xt}}\sum_{i=1}^{N_{u_0}}\sum_{j=1}^{N_{xt}}\left|G_{\theta}(u_0^i)(x_j, t_j) - u^i(x_j, t_j)\right|^2.
$$
To measure how good is the model in functions that it has never seen, we use the relative $L^2$ error in a test set:
$$
    \varepsilon = \frac{1}{N_{test}}\sum_{i=1}^{N_{test}}\frac{\|\hat{u}^i - u^i\|_2}{\|u^i\|_2}.
$$

## 1. Initial Configuration

Here we are only importing the basic libraries and the modules of the project which are in `src/`.

In [ ]:
# To import my files
import sys
import os

src = os.path.abspath("./src")

if src not in sys.path:
    sys.path.append(src)

In [ ]:
# Imports
from tqdm import tqdm
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

from src.model import DeepONet
from src.dataset import generate_data

## 2. Experiment Parameters

In this code we are considering the essential parameters of the project: `nx` and `nt` are the number of points of the grid in space and time, so each solution has `nx * nt` values, and `T` is the final time, the time domain is $[0, T]$; `m` is the number of sensor points where we evaluate the initial condition, `p` and `hidden` are the sizes of the DeepONet; `epochs`, `batch_size` and `lr` are the hyperparameters of the optimization; lastly, we are training the model in `device`.

In [ ]:
# Parameters
nx, nt = 100, 50
T = 0.1

# Architecture values
m, p, hidden = 100, 64, 64

# Train hyperparameters
epochs, batch_size, lr = 2500, 128, 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 3. Train and Test Data

Here we have the `generate_data` which generates three tensors: `u0` are the initial conditions evaluated in the sensors, `xt` are the points $(x, t)$ of the grid where we want the solution, and `y` are the exact solutions in that grid. We generate 3000 functions for train and 500 for test, with different seeds so they are different functions. Then we send everything to the `device` and we use a `DataLoader` to train with mini-batches of functions.

In [ ]:
# To generate data
u0_tr, xt, y_tr = generate_data(3000, 5, 100, 100, 50, alpha=1.0, T=0.1, seed=0)
u0_te, _,  y_te = generate_data(500,  5, 100, 100, 50, alpha=1.0, T=0.1, seed=1)

# Send data to the device
u0_tr_d, y_tr_d = u0_tr.to(device), y_tr.to(device)
u0_te_d, y_te_d = u0_te.to(device), y_te.to(device)
xt_d = xt.to(device)

loader = DataLoader(TensorDataset(u0_tr_d, y_tr_d), batch_size=batch_size, shuffle=True)

## 4. Model, Optimizer Algorithm and Train Loop

In this code we instance the `DeepONet` which is going to be trained with the Adam optimizer and we use the `MSELoss` as the training loss. We also add a learning rate scheduler `ExponentialLR` which multiplies the learning rate by a constant factor each epoch, with this factor the learning rate decays by $10^{-2}$ in total.

In [ ]:
model = DeepONet(m, p, hidden).to(device)
mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=(1e-2) ** (1 / epochs))

This function calculates the relative $L^2$ error that we defined at the beginning, it is evaluated in the test set and it is averaged over the functions.

In [ ]:
@torch.no_grad()
def rel_l2(model, u0, xt, y):
    """L2 relative error, averaged over the functions."""
    model.eval()
    pred = model(u0, xt)
    err = torch.linalg.norm(pred - y, dim=1) / torch.linalg.norm(y, dim=1)
    return err.mean().item()

This is basically the train loop. In each epoch we go through all the mini-batches, we calculate the loss against the exact solution and we update the weights. At the end of each epoch we save the mean train loss and the relative $L^2$ error in the test set. Here it is only interesting to know that this code took roughly 4 minutes using a GPU.

In [ ]:
# Train loop
history = {"train_loss": [], "test_rel_l2": []}

model.train()
for _ in tqdm(range(1, epochs + 1), desc="Train progress", unit=" Epoch"):
    running, n_batches = 0.0, 0
    for u0_b, y_b in loader:
        optimizer.zero_grad()
        
        pred = model(u0_b, xt_d)
        loss = mse_loss(pred, y_b)
        
        loss.backward()
        optimizer.step()
        
        running += loss.item()
        n_batches += 1
        
    scheduler.step()

    history["train_loss"].append(running / n_batches)
    history["test_rel_l2"].append(rel_l2(model, u0_te_d, xt_d, y_te_d))

## Results

First we can show the train loss and the test relative $L^2$ error per epoch, both in logarithm scale.

In [ ]:
# Train curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].semilogy(history["train_loss"])
ax[1].semilogy(history["test_rel_l2"], color="C1")

ax[0].set(xlabel="epoch", ylabel="MSE")
ax[0].set(xlabel="epoch", ylabel="Relative L2 Error")

plt.tight_layout()

ax[0].grid(alpha=0.4)
ax[1].grid(alpha=0.4)

plt.show()

Finally, we compare the exact solution and the one predicted by the DeepONet for a function of the test set, along with the absolute error between them. Remember that the model has never seen this initial condition during the training.

In [ ]:
model.eval()
with torch.no_grad():
    pred_te = model(u0_te_d, xt_d).cpu()
    
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Exact and predicted matrix
exact = y_te[0].reshape(nx, nt).numpy()
pred = pred_te[0].reshape(nx, nt).numpy()

# Absolute error
err = np.abs(pred - exact)
vmin, vmax = exact.min(), exact.max()

t_coord = np.linspace(0, T, nt)
x_coord = np.linspace(0, 1, nx)

# Heatmaps
im0 = axes[0].pcolormesh(t_coord, x_coord, exact, shading='nearest', vmin=vmin, vmax=vmax, cmap="viridis")
im1 = axes[1].pcolormesh(t_coord, x_coord, pred,  shading='nearest', vmin=vmin, vmax=vmax, cmap="viridis")
im2 = axes[2].pcolormesh(t_coord, x_coord, err,   shading='nearest', cmap="magma")

axes[0].set_title("Exact solution")
axes[1].set_title("DeepONet solution")
axes[2].set_title("Absolute error")

for col, im in enumerate([im0, im1, im2]):
    axes[col].set_xlabel(r"$t$")
    axes[col].set_ylabel(r"$x$")
    fig.colorbar(im, ax=axes[col])
    
plt.tight_layout()
plt.show()